In [5]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "scripts" else CWD
sys.path.insert(0, str(PROJECT_ROOT))

import finite_difference_quantum as fdq
from scipy.sparse.linalg import ArpackNoConvergence, eigs, eigsh

plt.rcParams.update(
    {
        "font.family": "serif",
        "mathtext.fontset": "stix",
        "axes.linewidth": 1.1,
        "grid.alpha": 0.25,
        "grid.linestyle": "--",
        "grid.linewidth": 0.8,
        "figure.dpi": 300,
        "savefig.dpi": 300,
    }
)


In [6]:
def analytical_isw_ground_energy(L: float) -> float:
    return float(np.pi**2 / (2.0 * (float(L) ** 2)))


def numerical_isw_ground_energy(*, N: int, order: int, L: float) -> float:
    def V0(x):
        return np.zeros_like(x)

    H = fdq.hamiltonian(V0, 0.0, float(L), int(N), order=int(order))

    if int(order) == 2:
        ev = eigsh(H, k=1, which="SA", tol=1e-10, maxiter=200000, return_eigenvectors=False)[0]
        return float(np.real(ev))

    # 4th-order boundary closures can make H slightly non-symmetric -> use eigs (shift-invert).
    try:
        ev = eigs(H, k=1, sigma=0.0, which="LM", tol=1e-10, maxiter=400000, return_eigenvectors=False)[0]
    except ArpackNoConvergence as e:
        if getattr(e, "eigenvalues", None) is not None and len(e.eigenvalues) > 0:
            ev = e.eigenvalues[0]
        else:
            raise

    return float(np.real(ev))


In [7]:
L = 10.0
N_min, N_max, N_points = 200, 6000, 60

# 60 log-spaced integer N values (guaranteed unique by oversampling)
raw = np.rint(np.logspace(np.log10(N_min), np.log10(N_max), N_points * 4)).astype(int)
N_values = np.unique(np.clip(raw, N_min, N_max))
N_values = N_values[np.linspace(0, len(N_values) - 1, N_points).astype(int)]

out_path = (PROJECT_ROOT / "images" / "isw_convergence.png").resolve()
print("Will write:", out_path)
print("N range:", int(N_values.min()), "..", int(N_values.max()), "(points:", len(N_values), ")")


Will write: /Users/joshhiller/Desktop/Thesis/images/isw_convergence.png
N range: 200 .. 6000 (points: 60 )


In [8]:
E_ref = analytical_isw_ground_energy(L)
print("Analytical ground energy:", E_ref)

errors_2nd = np.empty(len(N_values), dtype=float)
errors_4th = np.empty(len(N_values), dtype=float)

for i, N in enumerate(map(int, N_values)):
    # light progress updates (printing every iteration slows nbconvert)
    if i == 0 or (i + 1) % 10 == 0 or (i + 1) == len(N_values):
        print(f"[{i+1}/{len(N_values)}] N={N}")

    E2 = numerical_isw_ground_energy(N=N, order=2, L=L)
    E4 = numerical_isw_ground_energy(N=N, order=4, L=L)

    errors_2nd[i] = abs(E2 - E_ref) / abs(E_ref)
    errors_4th[i] = abs(E4 - E_ref) / abs(E_ref)

print("done")


Analytical ground energy: 0.04934802200544679
[1/60] N=200
[10/60] N=334
[20/60] N=595
[30/60] N=1060
[40/60] N=1889
[50/60] N=3366
[60/60] N=6000
done


In [9]:
fig = plt.figure(figsize=(7.6, 4.6))
ax = fig.add_subplot(1, 1, 1)

ax.plot(N_values, errors_2nd, "o-", color="#1f77b4", linewidth=2.0, markersize=5.5, label="2nd Order")
ax.plot(N_values, errors_4th, "s--", color="#ff7f0e", linewidth=2.0, markersize=5.5, label="4th Order")

ax.set_xscale("log")
ax.set_yscale("log")
ax.grid(True, which="both")

ax.set_xlabel("Grid Size N")
ax.set_ylabel("Relative Error (Ground State)")
ax.set_title("Convergence Analysis: ISW")
ax.legend(loc="upper right")

# extend x-limits a bit so the last point isn't on the frame
ax.set_xlim(N_values.min() * 0.9, N_values.max() * 1.1)

out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_path, bbox_inches="tight")
plt.close(fig)

print("Saved:", out_path)


Saved: /Users/joshhiller/Desktop/Thesis/images/isw_convergence.png
